In [1]:
import numpy as np
import pandas as pd


In [2]:
df_cleaned = pd.read_csv(r"C:\Users\dhanu\OneDrive\Desktop\data_science_project\1_datasets\teleco_dataset_formatted_2.csv")
df_cleaned.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,TotalCharges,Churn,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TotalCharges_log
0,0,0,1,0,1,0,0,0,1,0,...,29.85,0,0,0,0,0,0,1,0,3.429137
1,1,0,0,0,34,1,0,1,0,1,...,1889.50,0,0,0,1,0,0,0,1,7.544597
2,1,0,0,0,2,1,0,1,1,0,...,108.15,1,0,0,0,0,0,0,1,4.692723
3,1,0,0,0,45,0,0,1,0,1,...,1840.75,0,0,0,1,0,0,0,0,7.518471
4,0,0,0,0,2,1,0,0,0,0,...,151.65,1,1,0,0,0,0,1,0,5.028148


In [3]:
df_cleaned.columns


Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'InternetService_Fiber optic', 'InternetService_No',
       'Contract_One year', 'Contract_Two year',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'TotalCharges_log'],
      dtype='str')

In [4]:
# Feature Engineering

# 1 -> Total Number of Services or Add-ons subscribed

total_addons = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']

df_cleaned["TotalServices"] = df_cleaned[total_addons].sum(axis = 1)
df_cleaned["TechServices_Support"] = df_cleaned[service_cols].sum(axis = 1)

In [5]:
# 2 -> MonthlyCharges * Tenure vs TotalCharges

df_cleaned['AvgMonthlyCharges'] = df_cleaned['TotalCharges'] / np.where(df_cleaned['tenure'] == 0, 1, df_cleaned['tenure'])
df_cleaned['ChargeInflationRatio'] = df_cleaned["MonthlyCharges"] / (df_cleaned["AvgMonthlyCharges"] + 1e-5)


In [6]:
# 3 -> Total Number of Services taken and how they are charged accordingly

df_cleaned["CostperService"] = df_cleaned["MonthlyCharges"] / np.where(df_cleaned["TotalServices"] == 0, 1, df_cleaned["TotalServices"])


In [7]:
# 4 -> New Customer and Long Term Customer

df_cleaned["IsNewCust"] = (df_cleaned["tenure"] <= 12).astype(int)
df_cleaned["IsLongTermCust"] = (df_cleaned["tenure"] >= 48).astype(int)


In [8]:
# 5 -> Based on Customer Churn Risk

# From EDA, it was observed that those without a contract are highly likely to churn

df_cleaned["IsMonth_to_Month"] = ((df_cleaned["Contract_One year"] == 0) & (df_cleaned["Contract_Two year"] == 0)).astype(int)

df_cleaned["RiskyCustomerProfile"] = (
        (df_cleaned["IsMonth_to_Month"] == 1) & (df_cleaned["tenure"] <= 12) & (df_cleaned["PaymentMethod_Electronic check"] == 1)
).astype(int)

# 6 -> Based on Family

df_cleaned["HasFamily"] = ((df_cleaned["Partner"] == 1) | (df_cleaned["Dependents"] == 1)).astype(int)


In [9]:
# Total Charges and the log scale both exist, so removing Total Charges with log scale can be done to avoid redundancy

X = df_cleaned.drop(columns = ["TotalCharges", "Churn"])
Y = df_cleaned["Churn"]

X.to_csv(r"C:\Users\dhanu\OneDrive\Desktop\data_science_project\3_Feature_Eng & Data_Prep\Input_Output_Saved\X_INPUT_DATA", index= False)
Y.to_csv(r"C:\Users\dhanu\OneDrive\Desktop\data_science_project\3_Feature_Eng & Data_Prep\Input_Output_Saved\Y_OUTPUT_LABELS", index = False)